# Chicago Food Safety × 311 Sanitation Complaints

**Story 1:** Does neighborhood sanitation reflect restaurant food safety?  
**Story 3:** Are food safety risks being overlooked?

This notebook focuses on the analysis and charts. The input files are already cleaned. Run the cells from top to bottom.

**Note:** 311 complaints are shown as **counts**, not population-adjusted rates. Comparisons may be affected by neighborhood population and reporting behavior.

## Load the two cleaned datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from pathlib import Path

folder = Path('/Users/michael818/Desktop/BA780_dataCleaning/整理好的資料')
food = pd.read_csv(folder / 'Food_Inspections_Cleaned.csv', low_memory=False)
complaints = pd.read_csv(folder / '311_Sanitation_Cleaned_2010_2026.csv', low_memory=False)
output = folder / 'food_safety_visualizations'
output.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

## Prepare data for the two stories

We group both datasets by **ZIP Code and Year** because inspection records and 311 reports do not share an event ID. For Food Inspections, the failure rate is failed inspections divided by completed inspections. For 311, we count reports. A left join keeps areas with inspections even when no matching 311 report is recorded.

We focus on **2021–2025**, sum the five-year totals by ZIP Code, and use Chicago ZIP boundaries for the maps.

In [ ]:
# Food inspection results by ZIP and year
food['Zip'] = food['Zip'].astype('string').str.extract(r'(\d{5})')[0]
food['Year'] = pd.to_numeric(food['Year'], errors='coerce')
food['Results'] = food['Results'].str.strip().str.lower()
food = food[food['Results'].isin(['pass', 'fail', 'pass w/ conditions'])].copy()
food['Failed'] = (food['Results'] == 'fail').astype(int)
food_year = food.groupby(['Zip', 'Year'], as_index=False).agg(
    Inspection_Count=('Failed', 'size'), Fail_Count=('Failed', 'sum'))

# 311 sanitation reports by ZIP and year
complaints['Zip'] = complaints['ZIP Code'].astype('string').str.extract(r'(\d{5})')[0]
complaints['Year'] = pd.to_numeric(complaints['Year'], errors='coerce')
complaint_year = complaints.groupby(['Zip', 'Year']).size().reset_index(name='Complaint_Count')

# Join and summarize 2021–2025
joined = food_year.merge(complaint_year, on=['Zip', 'Year'], how='left')
joined['Complaint_Count'] = joined['Complaint_Count'].fillna(0)
joined = joined[joined['Year'].between(2021, 2025)]
summary = joined.groupby('Zip', as_index=False)[
    ['Inspection_Count', 'Fail_Count', 'Complaint_Count']].sum()
summary['Fail_Rate'] = summary['Fail_Count'] / summary['Inspection_Count'] * 100

# Chicago ZIP boundaries
chicago = gpd.read_file('https://data.cityofchicago.org/resource/unjd-c2ca.geojson?$limit=1000')
chicago['Zip'] = chicago['zip'].astype('string').str.extract(r'(\d{5})')[0]
chicago = chicago.drop_duplicates('Zip')
summary = summary[summary['Zip'].isin(chicago['Zip'])].copy()
analysis = summary[summary['Inspection_Count'] >= 30].copy()

## Story 1 — Does neighborhood sanitation reflect restaurant food safety?

**Chart 1: Side-by-side ZIP maps.** Compare the geographic patterns of food inspection failure rates and 311 sanitation complaint counts.

**Chart 2: Scatter plot with regression line.** Explore whether ZIP Codes with more complaints also tend to have higher inspection failure rates. **Correlation does not imply causation.**

In [ ]:
# Story 1 — Two maps
mapped = chicago.merge(summary, on='Zip', how='left')
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, column, title, color in [
    (axes[0], 'Fail_Rate', 'Food Inspection Failure Rate (%)', 'Blues'),
    (axes[1], 'Complaint_Count', '311 Sanitation Complaints (Total)', 'Oranges')]:
    mapped.plot(column=column, cmap=color, legend=True, ax=ax,
                edgecolor='white', linewidth=0.3)
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Chicago ZIP Codes | 2021–2025')
plt.tight_layout()
plt.savefig(output / 'story1_two_maps.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Story 1 — Scatter plot
plt.figure(figsize=(9, 6))
sns.regplot(data=analysis, x='Complaint_Count', y='Fail_Rate',
            scatter_kws={'alpha': 0.7})
plt.title('Story 1 | Complaints vs. Inspection Failure Rate')
plt.xlabel('311 Sanitation Complaints (2021–2025 Total)')
plt.ylabel('Food Inspection Failure Rate (%)')
plt.tight_layout()
plt.savefig(output / 'story1_scatter.png', dpi=300)
plt.show()
print('Pearson correlation:', round(analysis['Complaint_Count'].corr(analysis['Fail_Rate']), 3))

## Story 3 — Are food safety risks being overlooked?

**Chart 3: Quadrant scatter plot.** Split ZIP Codes into four groups using the **median** complaint count and median failure rate:

- Low complaints / High failures — potentially overlooked areas
- High complaints / High failures
- Low complaints / Low failures
- High complaints / Low failures

**Chart 4: Risk category map.** Show where the four groups are located in Chicago.

These are **descriptive categories, not confirmed risk levels**. Low complaint counts may reflect underreporting, and larger populations can generate more complaints.

In [ ]:
# Story 3 — Classify ZIP Codes into four groups
risk = analysis.copy()
x_mid = risk['Complaint_Count'].median()
y_mid = risk['Fail_Rate'].median()

risk['Category'] = np.select([
    (risk['Complaint_Count'] < x_mid) & (risk['Fail_Rate'] >= y_mid),
    (risk['Complaint_Count'] >= x_mid) & (risk['Fail_Rate'] >= y_mid),
    (risk['Complaint_Count'] < x_mid) & (risk['Fail_Rate'] < y_mid)
], [
    'Low complaints / High failures',
    'High complaints / High failures',
    'Low complaints / Low failures'
], default='High complaints / Low failures')

plt.figure(figsize=(10, 7))
sns.scatterplot(data=risk, x='Complaint_Count', y='Fail_Rate',
                hue='Category', s=85)
plt.axvline(x_mid, color='gray', linestyle='--')
plt.axhline(y_mid, color='gray', linestyle='--')
plt.title('Story 3 | Four Neighborhood Patterns')
plt.xlabel('311 Sanitation Complaints (2021–2025 Total)')
plt.ylabel('Food Inspection Failure Rate (%)')
plt.tight_layout()
plt.savefig(output / 'story3_quadrant_scatter.png', dpi=300)
plt.show()

In [ ]:
# Story 3 — Category map
risk_map = chicago.merge(risk[['Zip', 'Category']], on='Zip', how='left')
fig, ax = plt.subplots(figsize=(10, 10))
risk_map.plot(column='Category', categorical=True, legend=True,
              ax=ax, edgecolor='white', linewidth=0.3)
ax.set_title('Story 3 | Neighborhood Patterns in Chicago')
ax.axis('off')
plt.tight_layout()
plt.savefig(output / 'story3_category_map.png', dpi=300, bbox_inches='tight')
plt.show()